In [1]:
import pandas as pd

In [6]:
df=pd.read_csv(r'C:\Users\Administrator\Downloads\AutoML_Project\output\preprocessed_data.csv')

In [7]:
df.isnull().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
dtype: int64

In [9]:
for i in df.columns:
    if "UNKNOWN" in df[i].values:
        print(i)

HomePlanet
CryoSleep
Cabin
Destination
VIP
Name


In [10]:
df['HomePlanet'].value_counts()

HomePlanet
Earth      4602
Europa     2131
Mars       1759
UNKNOWN     201
Name: count, dtype: int64

In [11]:
df_test=pd.read_csv(r'C:\Users\Administrator\Downloads\AutoML_Project\output\test.csv')

In [12]:
df_test.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [13]:
df_test=df_test.set_index('PassengerId')

In [14]:
df_test

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
PassengerId,,,,,,,,,,,,
0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez
...,...,...,...,...,...,...,...,...,...,...,...,...
9266_02,Earth,True,G/1496/S,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,Jeron Peter
9269_01,Earth,False,NaN,TRAPPIST-1e,42.0,False,0.0,847.0,17.0,10.0,144.0,Matty Scheron
9271_01,Mars,True,D/296/P,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,Jayrin Pore


In [19]:
import sys
sys.path.append(r"C:\Users\Administrator\Downloads\AutoML_Project\src")

In [16]:
import pickle
import pandas as pd
import numpy as np


# -------------------------------------------------
# Apply saved preprocessing parameters
# -------------------------------------------------
def apply_preprocessing(df, params):

    X = df.copy()

    # Drop training-time removed columns
    X.drop(columns=params.get("drop_cols", []), errors="ignore", inplace=True)

    # Fill missing values
    for col in X.columns:

        if col in params.get("means", {}):
            X[col] = X[col].fillna(params["means"][col])

        elif col in params.get("fill_values", {}):
            X[col] = X[col].fillna(params["fill_values"][col])

    # Apply outlier clipping
    for col, bounds in params.get("outlier_bounds", {}).items():

        if col in X.columns:
            lower, upper = bounds
            X[col] = np.clip(X[col], lower, upper)

    return X


# -------------------------------------------------
# Apply saved encoders
# -------------------------------------------------
def apply_encoding(df, encoders):

    X = df.copy()

    for col, mapping in encoders.items():

        if col in X.columns:

            X[col] = X[col].astype(str)

            # If saved as dict mapping
            if isinstance(mapping, dict):

                X[col] = X[col].map(mapping).fillna(-1)

            # If saved as LabelEncoder
            else:

                vals = X[col]

                known = set(mapping.classes_)

                vals = vals.map(lambda x: x if x in known else "UNKNOWN")

                if "UNKNOWN" not in mapping.classes_:
                    mapping.classes_ = np.append(mapping.classes_, "UNKNOWN")

                X[col] = mapping.transform(vals)

    return X


# -------------------------------------------------
# Align columns safely
# -------------------------------------------------
def align_columns(df, required_cols):

    X = df.copy()

    for col in required_cols:
        if col not in X.columns:
            X[col] = np.nan

    return X[required_cols]


# -------------------------------------------------
# Main prediction function
# -------------------------------------------------
def predict(df, model_path):

    # Load artifact
    with open(model_path, "rb") as f:
        artifact = pickle.load(f)

    params = artifact["preprocessing_params"]
    encoders = artifact.get("encoders", {})
    models = artifact["models"]
    features = artifact["features"]
    selected_features = artifact["selected_features"]

    # Step 1: align raw columns
    X = align_columns(df, features)

    # Step 2: preprocessing
    X = apply_preprocessing(X, params)

    # Step 3: encoding
    X = apply_encoding(X, encoders)

    # Step 4: final selected features
    X = align_columns(X, selected_features)

    # Step 5: predictions
    preds = {}
    probas = {}

    for name, model in models.items():

        preds[name] = model.predict(X)

        if hasattr(model, "predict_proba"):
            probas[name] = model.predict_proba(X)[:, 1]

    pred_df = pd.DataFrame(preds)
    proba_df = pd.DataFrame(probas)

    return pred_df, proba_df


# -------------------------------------------------
# MAIN
# -------------------------------------------------
if __name__ == "__main__":

    test_path = r"C:\Users\Administrator\Downloads\AutoML_Project\output\test.csv"
    model_path = r"C:\Users\Administrator\Downloads\AutoML_Project\automl_full.pkl"

    output_pred = r"C:\Users\Administrator\Downloads\AutoML_Project\output\predictions.csv"
    output_prob = r"C:\Users\Administrator\Downloads\AutoML_Project\output\probabilities.csv"

    # Load test data
    df = pd.read_csv(test_path)

    # Predict
    pred_df, proba_df = predict(df, model_path)

    # Save outputs
    pred_df.to_csv(output_pred, index=False)
    proba_df.to_csv(output_prob, index=False)


    print("✅ Predictions saved successfully")
    # -----------------------------------------
    # Create separate file for each model
    # -----------------------------------------
    for model_name in pred_df.columns:

        final_df = pd.DataFrame({
            "PassengerId": df["PassengerId"],
            "Transported": pred_df[model_name]
        })
        output_folder = r"C:\Users\Administrator\Downloads\AutoML_Project\output"


        file_name = f"{output_folder}\\{model_name}_predictions.csv"

        final_df.to_csv(file_name, index=False)

        print(f"✅ Saved: {file_name}")

    print("✅ All model prediction files created")

✅ Predictions saved successfully
✅ Saved: C:\Users\Administrator\Downloads\AutoML_Project\output\Logistic Regression_predictions.csv
✅ Saved: C:\Users\Administrator\Downloads\AutoML_Project\output\Random Forest_predictions.csv
✅ Saved: C:\Users\Administrator\Downloads\AutoML_Project\output\Gradient Boosting_predictions.csv
✅ Saved: C:\Users\Administrator\Downloads\AutoML_Project\output\SVM_predictions.csv
✅ All model prediction files created


In [ ]:
df.head()
pred_df.head()

SyntaxError: invalid syntax (2819209152.py, line 1)

In [8]:
# Load artifact
df = pd.read_csv(r"C:\Users\Administrator\Downloads\AutoML_Project\output\test.csv")   # 🔁 change path if needed
model_path = r"C:\Users\Administrator\Downloads\AutoML_Project\automl_full.pkl"
# Load artifact
with open(model_path, "rb") as f:
    artifact = pickle.load(f)

params = artifact["preprocessing_params"]
# selector = artifact["selector"]
encoders = artifact["encoders"]

models = artifact["models"]
features = artifact["features"]

# Step 1: Align columns
X = df.copy()
X = X[features]

# Step 2: Apply preprocessing
X = apply_preprocessing(X, params)

# Step 3: encoding
X = apply_encoding(X, encoders)

# Step 3: Feature selection
X = X[artifact["selected_features"]]

In [9]:
X

,Age,Spa,RoomService,CryoSleep,VRDeck,FoodCourt,HomePlanet,ShoppingMall
0,27.00000,0.0,0.0,1,0.0,0.0,Earth,0.0
1,19.00000,222.5,0.0,0,0.0,9.0,Earth,0.0
2,31.00000,0.0,0.0,1,0.0,0.0,Europa,0.0
3,38.00000,181.0,0.0,0,177.5,295.0,Europa,0.0
4,20.00000,0.0,10.0,0,0.0,0.0,Earth,112.5
...,...,...,...,...,...,...,...,...
4272,34.00000,0.0,0.0,1,0.0,0.0,Earth,0.0
4273,42.00000,10.0,0.0,0,144.0,295.0,Earth,17.0
4274,28.82793,0.0,0.0,1,0.0,0.0,Mars,0.0
4275,28.82793,0.0,0.0,0,177.5,295.0,Europa,0.0
